# Data Cleaning
It is apparant that we cannot exactly work with the data that has been provided. So data cleaning is must!

In [101]:
import pandas as pd
import numpy as np

In [102]:
# create new dataframe called cleaned data
cleaned_data = {}

# load raw data
raw_data = pd.read_csv("../../dataset/raw_data.csv", usecols=[0, 1, 3, 6, 8, 9, 12, 15, 16, 17, 18, 19, 20, 21, 22, 23])
# now we drop rows where cutoff is true. because cutoff only tells us that the journey didnt end, but crossed the threshold of the max time upto which gps can measure.
# since we have total time and distance with us, the segment time and distance do not matter
raw_data = raw_data[raw_data["is_cutoff"] == False]
print(raw_data.info)

<bound method DataFrame.info of             data          trip_creation_time route_type  \
4       training  2018-09-20 02:35:36.476840    Carting   
9       training  2018-09-20 02:35:36.476840    Carting   
14      training  2018-09-23 06:42:06.021680        FTL   
16      training  2018-09-14 15:42:46.437249    Carting   
18      training  2018-09-13 20:44:19.424489        FTL   
...          ...                         ...        ...   
144846  training  2018-09-26 19:50:29.657378        FTL   
144848  training  2018-09-14 18:45:34.164734    Carting   
144853  training  2018-09-22 11:30:41.399439        FTL   
144857  training  2018-09-17 11:35:28.838714    Carting   
144866  training  2018-09-20 16:24:28.436231    Carting   

                              source_name                   destination_name  \
4              Anand_VUNagar_DC (Gujarat)      Khambhat_MotvdDPP_D (Gujarat)   
9           Khambhat_MotvdDPP_D (Gujarat)         Anand_Vaghasi_IP (Gujarat)   
14      Bhiwandi_Ma

## list of parameters to be included in cleaned data
0. data_type (test/train)
1. centers as nodes
2. is_carting 
3. is_ftl
4. start_hour (0-23)
5. start_day_of_week (0-6)
6. osrm time
7. osrm distance
8. actual time
9. actual distance

In [103]:
raw_data = raw_data.dropna()

In [104]:
# converting all hub names into numberred format (for graph nodes)
all_hubs_list = pd.concat([raw_data["source_name"], raw_data["destination_name"]]).unique()
dict_hub_names_to_id = {hub_name: idx for idx, hub_name in enumerate(all_hubs_list)}


In [105]:
# start creating cleaned data
cleaned_data = {}
cleaned_data["data"] = (raw_data["data"]).tolist()
cleaned_data["source_number"] = map(lambda x: dict_hub_names_to_id[x], raw_data["source_name"])
cleaned_data["destination_name"] = map(lambda x: dict_hub_names_to_id[x], raw_data["destination_name"])
is_carting = list(map(lambda x: 1 if x=="Carting" else 0, raw_data["route_type"].tolist()))
cleaned_data["is_carting"] = is_carting
is_ftl = list(map(lambda x: 1 if x=="FTL" else 0, raw_data["route_type"].tolist()))
cleaned_data["is_ftl"] = is_ftl

In [106]:
raw_data["od_start_time"] = pd.to_datetime(raw_data["od_start_time"])
cleaned_data["day_of_week"] = (raw_data["od_start_time"].dt.day_of_week).tolist()
cleaned_data["start_hour"] = (raw_data["od_start_time"].dt.hour).tolist()
cleaned_data["is_day"] = list(map(lambda x: 1 if (x>=8 and x <= 19) else 0, cleaned_data["start_hour"]))
cleaned_data["is_night"] = list(map(lambda x: 1 if not(x>=8 and x <= 19) else 0, cleaned_data["start_hour"]))
cleaned_data["is_cutoff"] = list(map(lambda x: 1 if x=="True" else 0, raw_data["is_cutoff"].tolist())) 
cleaned_data["osrm_time"] = raw_data["osrm_time"].tolist()
cleaned_data["osrm_distance"] = raw_data["osrm_distance"].tolist()
cleaned_data["actual_distance"] = raw_data["actual_distance_to_destination"].tolist()
cleaned_data["actual_time"] = raw_data["actual_time"].tolist()
cleaned_data["real_actual_time_factor"] = raw_data["factor"].tolist()


In [107]:
df = pd.DataFrame(cleaned_data)
df.to_csv("../../dataset/cleaned_data.csv")
print(df)

           data  source_number  destination_name  is_carting  is_ftl  \
0      training              0                 1           1       0   
1      training              1               504           1       0   
2      training              2                57           0       1   
3      training              3               263           1       0   
4      training              4                 5           0       1   
...         ...            ...               ...         ...     ...   
25975  training            499                31           0       1   
25976  training           1122                 2           1       0   
25977  training              2                57           0       1   
25978  training             17               303           1       0   
25979  training             14                30           1       0   

       day_of_week  start_hour  is_day  is_night  is_cutoff  osrm_time  \
0                3           3       0         1          0  